# 🤖 AI Resume Screening & Job Match System
### Exploratory Data Analysis + Model Evaluation

**Author:** Your Name  
**Dataset:** Custom Job Descriptions + Demo Resumes  
**Goal:** Build an NLP-powered resume screener that matches candidates to jobs

---
## Table of Contents
1. [Setup & Imports](#1)
2. [Dataset Overview](#2)
3. [Exploratory Data Analysis](#3)
4. [Skill Extraction](#4)
5. [Similarity Scoring](#5)
6. [Model Evaluation](#6)
7. [Insights & Conclusions](#7)

## 1. Setup & Imports <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_colwidth', 100)
print('✅ All imports successful')

## 2. Dataset Overview <a id='2'></a>

In [ ]:
df = pd.read_csv('../dataset/job_descriptions.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Experience Distribution ===')
print(df['experience_years'].describe())

## 3. Exploratory Data Analysis <a id='3'></a>

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Job Dataset EDA', fontsize=16, fontweight='bold')

# Experience distribution
axes[0,0].hist(df['experience_years'], bins=6, color='#667eea', edgecolor='white', linewidth=1.2)
axes[0,0].set_title('Experience Years Distribution')
axes[0,0].set_xlabel('Years')
axes[0,0].set_ylabel('Count')

# Location distribution
loc_counts = df['location'].value_counts()
axes[0,1].barh(loc_counts.index, loc_counts.values, color='#764ba2')
axes[0,1].set_title('Jobs by Location')
axes[0,1].set_xlabel('Count')

# Description length
df['desc_length'] = df['description'].str.len()
axes[1,0].hist(df['desc_length'], bins=10, color='#f093fb', edgecolor='white')
axes[1,0].set_title('Job Description Length')
axes[1,0].set_xlabel('Characters')

# Skills count per job
df['skill_count'] = df['required_skills'].apply(lambda x: len(x.split(',')))
axes[1,1].bar(range(len(df)), df['skill_count'], color='#4facfe')
axes[1,1].set_title('Skills Required per Job')
axes[1,1].set_xlabel('Job Index')
axes[1,1].set_ylabel('Skill Count')

plt.tight_layout()
plt.savefig('../screenshots/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Most in-demand skills across all jobs
all_skills = []
for skills_str in df['required_skills']:
    all_skills.extend([s.strip() for s in skills_str.split(',')])

skill_freq = Counter(all_skills).most_common(20)
skills_df = pd.DataFrame(skill_freq, columns=['Skill', 'Frequency'])

plt.figure(figsize=(12, 6))
bars = plt.barh(skills_df['Skill'][::-1], skills_df['Frequency'][::-1],
                color=plt.cm.viridis(np.linspace(0.2, 0.9, 20)))
plt.title('Top 20 Most In-Demand Skills', fontsize=14, fontweight='bold')
plt.xlabel('Frequency')
for bar, val in zip(bars, skills_df['Frequency'][::-1]):
    plt.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
             str(val), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../screenshots/top_skills.png', dpi=150, bbox_inches='tight')
plt.show()
print(skills_df.to_string(index=False))

## 4. Skill Extraction <a id='4'></a>

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from model.skill_extractor import extract_skills, extract_experience_years, extract_name

DEMO_RESUME = """
John Smith
john.smith@email.com

Data Scientist with 3 years of experience in machine learning and NLP.

SKILLS
Python, Machine Learning, Deep Learning, NLP, TensorFlow, PyTorch, Scikit-learn,
Pandas, NumPy, SQL, Tableau, Docker, AWS, Transformers, BERT, Spacy, Statistics

EXPERIENCE
Senior Data Scientist | TechCorp | 2021 - Present (3 years of experience)
- Built NLP models for text classification
- Deployed ML models on AWS using Docker
"""

extracted_skills = extract_skills(DEMO_RESUME)
extracted_exp    = extract_experience_years(DEMO_RESUME)
extracted_name   = extract_name(DEMO_RESUME)

print(f'Name: {extracted_name}')
print(f'Experience: {extracted_exp} years')
print(f'Skills ({len(extracted_skills)}): {extracted_skills}')

In [ ]:
# Visualize extracted vs total skills
total_unique = len(set(all_skills))
extracted_count = len(extracted_skills)

fig, ax = plt.subplots(figsize=(8, 5))
categories = ['Total Unique Skills\nin Dataset', 'Skills Extracted\nfrom Demo Resume']
values = [total_unique, extracted_count]
colors = ['#667eea', '#f093fb']
bars = ax.bar(categories, values, color=colors, width=0.4, edgecolor='white', linewidth=2)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', fontsize=14, fontweight='bold')
ax.set_title('Skill Extraction Results', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../screenshots/skill_extraction.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Similarity Scoring <a id='5'></a>

In [ ]:
from model.scorer import rank_jobs, compute_skill_match, compute_semantic_similarity

results = rank_jobs(DEMO_RESUME, extracted_skills, extracted_exp, df)
print('Top 10 Job Matches:')
display_cols = ['job_title', 'company', 'match_score', 'skill_score', 'semantic_score', 'experience_score']
print(results[display_cols].head(10).to_string(index=False))

In [ ]:
# Score distribution visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Scoring Distribution Across All Jobs', fontsize=14, fontweight='bold')

score_cols = ['skill_score', 'semantic_score', 'match_score']
titles = ['Skill Match Score', 'Semantic Score', 'Final Match Score']
colors = ['#667eea', '#f093fb', '#4facfe']

for ax, col, title, color in zip(axes, score_cols, titles, colors):
    ax.hist(results[col], bins=10, color=color, edgecolor='white', linewidth=1.2)
    ax.axvline(results[col].mean(), color='red', linestyle='--', label=f'Mean: {results[col].mean():.1f}%')
    ax.set_title(title)
    ax.set_xlabel('Score (%)')
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.savefig('../screenshots/score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
score_df = results[['skill_score', 'semantic_score', 'experience_score', 'match_score']]
corr = score_df.corr()

plt.figure(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Score Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../screenshots/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Model Evaluation <a id='6'></a>

In [ ]:
# Simulate ground truth for evaluation
# Jobs with 'data' or 'ml' in title are considered relevant for our Data Scientist resume
relevant_titles = ['Data Scientist', 'Machine Learning Engineer', 'NLP Engineer',
                   'Data Analyst', 'Data Engineer', 'AI Research Scientist',
                   'Business Intelligence Analyst']

results['is_relevant'] = results['job_title'].isin(relevant_titles).astype(int)
results['predicted_relevant'] = (results['match_score'] >= 40).astype(int)

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

precision = precision_score(results['is_relevant'], results['predicted_relevant'], zero_division=0)
recall    = recall_score(results['is_relevant'], results['predicted_relevant'], zero_division=0)
f1        = f1_score(results['is_relevant'], results['predicted_relevant'], zero_division=0)

print(f'Precision : {precision:.3f}')
print(f'Recall    : {recall:.3f}')
print(f'F1 Score  : {f1:.3f}')

cm = confusion_matrix(results['is_relevant'], results['predicted_relevant'])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Relevant', 'Relevant'],
            yticklabels=['Not Relevant', 'Relevant'])
plt.title('Confusion Matrix (threshold=40%)', fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../screenshots/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Threshold analysis
thresholds = range(10, 90, 5)
metrics = []
for t in thresholds:
    pred = (results['match_score'] >= t).astype(int)
    p = precision_score(results['is_relevant'], pred, zero_division=0)
    r = recall_score(results['is_relevant'], pred, zero_division=0)
    f = f1_score(results['is_relevant'], pred, zero_division=0)
    metrics.append({'threshold': t, 'precision': p, 'recall': r, 'f1': f})

metrics_df = pd.DataFrame(metrics)
plt.figure(figsize=(10, 5))
plt.plot(metrics_df['threshold'], metrics_df['precision'], 'b-o', label='Precision', linewidth=2)
plt.plot(metrics_df['threshold'], metrics_df['recall'],    'r-s', label='Recall',    linewidth=2)
plt.plot(metrics_df['threshold'], metrics_df['f1'],        'g-^', label='F1 Score',  linewidth=2)
plt.axvline(40, color='gray', linestyle='--', alpha=0.7, label='Threshold=40')
plt.xlabel('Match Score Threshold (%)')
plt.ylabel('Score')
plt.title('Precision / Recall / F1 vs Threshold', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../screenshots/threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Insights & Conclusions <a id='7'></a>

In [ ]:
print('=== KEY INSIGHTS ===')
print(f'\n1. Top matched job: {results.iloc[0]["job_title"]} @ {results.iloc[0]["company"]} ({results.iloc[0]["match_score"]:.1f}%)')
print(f'2. Jobs with >50% match: {(results["match_score"] > 50).sum()} / {len(results)}')
print(f'3. Average match score: {results["match_score"].mean():.1f}%')
print(f'4. Skill extraction found {len(extracted_skills)} skills from resume')
print(f'5. Most common missing skill: {results["missing_skills"].explode().value_counts().index[0]}')
print('\n=== SCORING FORMULA ===')
print('Final Score = 0.50 × Skill Match + 0.35 × Semantic Similarity + 0.15 × Experience')
print('\n=== SYSTEM CAPABILITIES ===')
print('✅ PDF text extraction with pdfplumber')
print('✅ Rule-based skill extraction from 80+ skills database')
print('✅ TF-IDF + Sentence Transformers semantic similarity')
print('✅ Experience year extraction via regex')
print('✅ Weighted composite scoring')
print('✅ Skill gap analysis & learning recommendations')
print('✅ Interactive Streamlit dashboard')